In [58]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view #for easy minima without using for loops
from math import *
from collections import deque
import sys
np.set_printoptions(threshold=sys.maxsize)

## Homework 2: Dodge FI2026
Instead of previous hyper-efficient code (which took too much time to optimize), put less effort into optimization, just make it work

O Earth FI2026
q [au] 9.78969676e-01 0.74161536
e [ ] 1.83889630e-02 0.19293483
i [rad] 7.55697209e-05 0.05877868
ω [rad] 5.34322623 2.22509624
Ω [rad] 2.63906227 3.55784498
M [rad] 1.88972281 3.64624167

In [11]:
# constants taken from Prof. Eggl's repository, and verified by AE402 code
mu_sun   = 1.32712440018e11   # km^3/s^2
mu_earth = 398600.4418        # km^3/s^2
R_earth  = 6378.137           # km
deg2rad  = np.pi/180.0
au2km = 149597870.7

In [42]:
print(f"{R_earth/au2km:.6f}")

0.000043


In [95]:
# Earth keplerian elements wrt sun
# q (periapsis), e, i, argpe, raan, Mean Anomaly (rad)
earth_kep_hel = [9.78969676e-01, 1.83889630e-02, 7.55697209e-05, 5.34322623, 2.63906227, 1.88972281]
fi2026_kep_hel = [0.74161536, 0.19293483, 0.05877868, 2.22509624, 3.55784498, 3.64624167]

# make it clearer to myself what I'm putting into each function
q1 = earth_kep_hel[0] *au2km
e1 = earth_kep_hel[1]
i1 = earth_kep_hel[2]
argpe1 = earth_kep_hel[3]
raan1 = earth_kep_hel[4]
M1 = earth_kep_hel[5]
a1 = q1/(1-e1) *au2km

q2 = fi2026_kep_hel[0] *au2km
e2 = fi2026_kep_hel[1]
i2 = fi2026_kep_hel[2]
argpe2 = fi2026_kep_hel[3]
raan2 = fi2026_kep_hel[4]
M2 = fi2026_kep_hel[5]
a2 = q2/(1-e2) *au2km

### PRoblem 1: Time to impact

In [96]:
# Let's not use any of the old code, I don't trust the MOID transformations to be right
# Need to get cartesian coords -> need cartesian timestepping equations corresponding to orb elements
# Newton's method for E
# Looks like I need Curtis to propagate in universal variables (Example 3.6, section 3.7, page 139)
# I want to step both Earth and FI2026's trajecs to where their minimum distance is below Earth's radius, and then take that time.

#for part 2, don't forget to calculate if FI2026 is in hill sphere

# two ways of doing this: either you find M(t) and continuously convert into cartesian, or you do Curtis propagation
# Either way requires a cartesian solver
# Simpler to find M(t) = sqrt(mu/a^3)*t, where t is time since periapsis
# t0 can be found, then at every timestep t, calculate r and find distance between.

# algorithm 3.3 (Curtis)
# note that r0 and v_r0 do NOT have to be at periapsis, they can be at any specified true anomaly
# alpha = 1/a, reciprocal of SMA

def solve_kep(M,e,tol = 1e-12,maxiter=200):
    """
    Updated from AE402 with reference to Prof. Eggl's code (faster loop function)
    M = E-e*sin(E) for elliptic orbits
    """
    E = M if e<0.8 else pi
    # print("Newton's Method:")
    # for _ in range(maxiter):
    for i in range(maxiter):
        g = E-e*sin(E) - M
        dg_dE = 1-e*cos(E) #derivative dg over dE
        dE = g/dg_dE
        E -= dE #condensed for efficiency
        print(dE)
        if abs(dE) < tol: #little updates to the rate instead of the value of M
            print("iter=",i) #debugging this stupid fxn
            break
        
        # # Original logic for this function:
        # E -= g/dg_dE #not at the same time as the dE limiter
        # if abs(g) > tol: #tol should be 1e-10 as that worked in 402
        #     break
    return E

def R3(theta):
    """Rotate around Z axis, now with rotating in correct direction this time"""
    c,s = np.cos(theta),np.sin(theta)
    return np.array([[ c, -s, 0],
                     [ s,  c, 0],
                     [ 0,  0, 1]])

def R1(theta):
    """Rotate around X axis, now with rotating in correct direction this time"""
    c,s = np.cos(theta),np.sin(theta)
    return np.array([[ 1, 0, 0],
                     [ 0,  c, -s],
                     [ 0,  s,  c]])

def keplerian_to_cartesian(p,a,e,i,argpe,raan,M,mu):
    """
    Convert from keplerian to cartesian, elliptical only
    Inputs:
    - p: Periapsis
    - a: SMA
    - e: eccentricity
    - i: inclination
    - argpe: argument of periapsis
    - raan: right ascension
    - M: mean anomaly
    - mu: grav parameter of orbiting body
    """
    E = solve_kep(M,e) #eccentric anomaly
    r = a*(1-e*cos(E)) #current radius from orbiting body
    
    #tan(f/2) = sqrt((1+e)/(1-e))*tan(E/2)
    f = 2*atan2(sqrt(1+e)*sin(E/2),sqrt(1-e)*cos(E/2)) #true anomaly
    print(f"Periapsis check: {r*(1+e*cos(f)):.4e}, Radius: {p/(1+e*cos(f)):.4e}, Actual radius: {r:.4e}") #check for periapsis
    r_pqw = np.array([r*cos(f), r*sin(f), 0])
    v_pqw = sqrt(mu/p)*np.array([-sin(f), e+cos(f), 0])
    # h2_div_mu = a*(1-e**2) #Reverse the SMA equation
    # r = h2_div_mu/(1+e*cos(theta))
    Q = R3(raan) @ R1(i) @ R3(argpe)
    r_vec = Q@r_pqw
    v_vec = Q@v_pqw
    return r_vec,v_vec

Brute force: Generate M from t, step t to get impact location

In [ ]:
t0_1 = M1/sqrt(mu_sun/a1**3) #time at initial M1
t0_2 = M2/sqrt(mu_sun/a2**3)
print(t0_1,t0_2)
print(t0_1/86400,t0_2/86400) #WTF!!!!

1.729669926348078e+19 2.951691981933508e+19
200193278512509.03 341631016427489.4


In [97]:
keplerian_to_cartesian(q1,a1,e1,i1,argpe1,raan1,M1,mu_sun)

-0.017361551316754636
2.610739087884235e-06
5.892832989281118e-14
iter= 2
Periapsis check: 2.2312e+16, Radius: 1.4739e+08, Actual radius: 2.2455e+16


(array([-1.98975036e+16, -1.04069492e+16,  1.41344310e+12]),
 array([ 1.34025867e+01, -2.67455036e+01,  1.28344462e-03]))

In [71]:
# t0_1 = M1/sqrt(mu_sun/a1**3) #time at initial M1
# t0_2 = M2/sqrt(mu_sun/a2**3)
# print(t0_1,t0_2) #seems suspiciously low... maybe we're at periapsis??

# dt = 1.0 #day
# r0_1, v0_1 = keplerian_to_cartesian(q1,a1,e1,i1,argpe1,raan1,M1,mu_sun)
# r0_2, v0_2 = keplerian_to_cartesian(q2,a2,e2,i2,argpe2,raan2,M2,mu_sun)

def minima(D_sq): # I lied about reusing code. Minima function is needed to filter the huge amount of data
    """
    Strategy:
    1) Append 0th element to end of D_sq (for wraparound purposes)
    2) Sliding window view to create giant array of n x 3
    3) Find minima
    4) Knowing center column is either [1: (n+1)] in np.append or D_sq itself in np.pad,
    subtract D_sq from that and find zeros. Indices that are 0 are local minima.
    5) Return indices for easy math
    """

    D_sq_wrap = np.pad(D_sq,(1,1),'wrap')
    q = sliding_window_view(D_sq_wrap,3) # Produces n-1 size array, slice first element
    # print(np.min(q,axis=1)-D_sq)
    return np.argwhere((np.min(q,axis=1)-D_sq) == 0 ).T[0] #Returns indices, needed to find L and nu

def time_to_moid(q1,a1,e1,i1,argpe1,raan1,M1, q2,a2,e2,i2,argpe2,raan2,M2, dt=0.1,dt_refine=1e-3):
    t0_1 = M1/sqrt(mu_sun/a1**3) #time at initial M1
    t0_2 = M2/sqrt(mu_sun/a2**3)
    t1 = t0_1 # Periapsis times are different, start at different times (same epoch)
    t2 = t0_2 
    # r0_1, v0_1 = keplerian_to_cartesian(q1,a1,e1,i1,argpe1,raan1,M1,mu_sun)
    # r0_2, v0_2 = keplerian_to_cartesian(q2,a2,e2,i2,argpe2,raan2,M2,mu_sun)
    dist = []
    deltat = []
    for i in range(int(365/dt)): # look ahead a year in advance
        M1_i = sqrt(mu_sun/a1**3)*t1 #Different M for different orbit
        M2_i = sqrt(mu_sun/a2**3)*t2
        r1_i, _ = keplerian_to_cartesian(q1,a1,e1,i1,argpe1,raan1,M1_i,mu_sun)
        r2_i, _ = keplerian_to_cartesian(q2,a2,e2,i2,argpe2,raan2,M2_i,mu_sun)
        print(np.linalg.norm(r2_i-r1_i))
        dist.append(np.linalg.norm(r2_i-r1_i))
        deltat.append(dt*86400*i)
        t1 += dt*86400 #dt is in days
        t2 += dt*86400
    min_dist_i = minima(dist)
    min_dist = np.array(dist)[minima(dist)]
    # min_dist_refine_refined_i = []
    min_dist_refine_refined = []
    deltat_refine_refined = []
    # for index in min_dist_i:
    #     t1 = t0_1 + deltat[index-2] #go back few days to check for local minima
    #     t2 = t0_2 + deltat[index-2]
    #     dist_refine = []
    #     # deltat_refine = []
    #     for j in range(int((10*dt)/dt_refine)): # look ahead till other side of trough
    #         M1_i = sqrt(mu_sun/a1**3)*t1 #Different M for different orbit
    #         M2_i = sqrt(mu_sun/a2**3)*t2
    #         r1_i, _ = keplerian_to_cartesian(q1,a1,e1,i1,argpe1,raan1,M1_i,mu_sun)
    #         r2_i, _ = keplerian_to_cartesian(q2,a2,e2,i2,argpe2,raan2,M2_i,mu_sun)
    #         # print(np.linalg.norm(r2_i-r1_i))
    #         dist_refine.append(np.linalg.norm(r2_i-r1_i))
    #         # deltat_refine.append(dt_refine*86400*j)
    #         t1 += dt_refine*86400 #dt is in days
    #         t2 += dt_refine*86400
    #     min_dist_refine_i = minima(dist_refine)
    #     min_dist_refine = np.array(dist_refine)[min_dist_refine_i]
    #     min_index = np.argmin(min_dist_refine)
    #     # min_dist_refine_refined_i.append(min_index)
    #     min_dist_refine_refined.append(min_dist_refine[min_index])
    #     deltat_refine_refined.append(deltat[index-2]+dt_refine*86400*min_index)
    return dist, min_dist_i, min_dist, np.min(np.array(dist)[minima(dist)]), min_dist_refine_refined, deltat_refine_refined

fi, fee, foe, fum, min_dist, deltat = time_to_moid(q1,a1,e1,i1,argpe1,raan1,M1, q2,a2,e2,i2,argpe2,raan2,M2)
print(fee)
print(foe)
print(np.argmin(foe))
print(fum)
print(foe[np.argmin(foe)])
print(fi[3597:3630])

1.0097499207343157
1.276165544664871
0.6506293858992509
0.9830872291555701
0.9268963671372988
0.630323598516535
1.016720813815067
0.8344111504063512
0.5112959332487075
0.9762639310696792
1.300000053722911
1.5867321748967416
1.022830503530838
0.6746962150797886
1.7109451930849522
0.9704008154112826
1.0935665301638129
1.9204561472339445
1.027822441603174
1.185900734767079
1.877484684095374
0.9657805809818667
0.6099331142205054
1.4505739208859267
1.0314904803436202
1.264644847469388
1.3447086851202161
0.9626292779256539
0.9574954429430841
0.13127253754138946
1.0336848753069943
0.803924320400281
0.25210921768927147
0.961103065004457
1.3043861406705832
0.9256507871253997
1.0343170473081174
0.6981901240609273
1.4415326180036925
0.9612778898553792
1.0671069383284852
1.5756228080824952
1.0333615591134206
1.2054092103763439
2.0582341954661723
0.9631450720846334
0.6024325415196264
1.6876806033939766
1.030856758893772
1.2513871907412861
1.833980356323857
0.9666115875970288
0.9875375095178807
0.84

In [64]:
print(np.array(deltat)/86400)
print(min_dist)
print(np.min(min_dist))
print(np.array(deltat[np.argmin(min_dist)])/86400)

[366.879  11.793  20.925  31.004  41.682  46.342  52.687  61.69   70.094
  83.834  93.314 102.317 112.396 118.731 123.862 133.941 142.944 153.023
 161.556 174.568 184.647 186.973 193.791 203.882 215.369 226.188 227.682
 233.759 246.867 255.106 260.139 265.925 276.076 284.315 295.802 300.095
 305.953 316.032 325.011 338.338 343.456 345.909 356.728]
[0.011309060793979888, 0.01235891766646863, 0.013149236849955278, 0.003216414429764364, 0.0075613078965507344, 0.016823337139845025, 0.017989938388195632, 0.0036945111277620077, 0.005865254492719899, 0.01860541331533968, 0.015062858244810046, 0.004578392111382676, 0.009597917688042695, 0.010522907960193909, 0.015490123999377072, 0.012961204353756509, 0.008979876004212062, 0.006133745541761223, 0.009253557757986895, 0.011966469659646798, 0.013068985210780296, 0.015320783365629398, 0.01049058277755253, 0.011324953739336377, 0.013751867418244898, 0.016618012629248797, 0.009553678690194446, 0.009343682464261161, 0.006329139689448834, 0.0125790369

In [8]:
%timeit keplerian_to_cartesian(q1,a1,e1,i1,argpe1,raan1,M1,mu_sun)

11.4 µs ± 61.8 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [9]:
%timeit keplerian_to_cartesian(q2,a2,e2,i2,argpe2,raan2,M2,mu_sun)

11.6 µs ± 175 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
